# 01 — Loading and the Three Sample-Size Bases

Establishes the reporting bases every later notebook depends on. Three different
denominators appear in the paper, and conflating them is the easiest way to
misquote a result:

| Base | N | What it is | Used for |
|---|---|---|---|
| **Full** | 7,000 | 5 languages × 1,400 annotated segments | COMET / TP / IP means, ANOVA, Welch's *t* |
| **Working** | 6,995 | Segments carrying a *numeric* human score | Every correlation with the human score |
| **MAR severity** | 1,258 | Marathi segments with a non-`Default` severity label | The severity-inversion analysis |

**Input:** `../data/indic/indic_parity_xlmr.xlsx`
**Output:** `../results/tables/sample_sizes.csv`

Every numerical value printed below is asserted against `../paper_numbers.yaml`
in notebook 12 and in `tests/test_paper_numbers.py`.

## Step 0 — Configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# ── Paths (relative; nothing in this repository uses an absolute path) ───────
DATA_XLMR   = Path("../data/indic/indic_parity_xlmr.xlsx")
DATA_MULTI  = Path("../data/indic/indic_parity_multi_tokenizer.xlsx")
DATA_LATIN  = Path("../data/latin/wmt24_ende_enes_metrics.xlsx")
TABLES_DIR  = Path("../results/tables")
FIGURES_DIR = Path("../results/figures")
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Sheet names exactly as they appear in the workbook ───────────────────────
SHEET_MAP = {
    "GUJ": "Indic_mt _for_analysis - Gujara",
    "TAM": "Indic_mt _for_analysis - Tamil_",
    "MAL": "Indic_mt _for_analysis - Malaya",
    "MAR": "Indic_mt _for_analysis - Marath",
    "HIN": "Indic_mt _for_analysis - Hindi_",
}

# ── Language display order (fixed throughout the paper) ──────────────────────
LANG_ORDER = ["GUJ", "TAM", "MAL", "MAR", "HIN"]

# ── Column names (as in the xlsx) ────────────────────────────────────────────
COL_COMET_NAT = "COMET"                                          # native-script COMET
COL_COMET_ROM = "COMET_romanized"                                # romanised COMET
COL_TP_NAT    = "Translation_xlmr_TP"                            # TP, native
COL_TP_ROM    = "Translation_Transliteration_romanized_xlmr_TP"  # TP, romanised
COL_IP_NAT    = "Translation_xlmr_IP"                            # IP, native
COL_IP_ROM    = "Translation_Transliteration_romanized_xlmr_IP"  # IP, romanised
COL_HUMAN     = "Human_scores"                                   # MQM-derived human score
COL_SEVERITY  = "Error1_Severity"                                # primary error severity

# ── Seeds (every stochastic step in this repository) ─────────────────────────
SEED_SPLIT = 42   # 50/50 within-language train/test split
SEED_GBM   = 0    # GradientBoostingRegressor
SEED_PERM  = 0    # paired permutation test

print("Config loaded. DATA_XLMR:", DATA_XLMR)

## Step 1 — Load the Workbook

The workbook stores one sheet per language. Each sheet is read whole; the
working set is derived by coercing the human-score column to numeric and
dropping the rows that fail to convert.

In [ ]:
def load_sheets(path):
    """Read the five per-language sheets.

    Returns two dicts keyed by ISO code:
      full  — all 1,400 rows per language (the 7,000-segment base)
      work  — rows carrying a numeric human score (the 6,995-segment base)

    Coercing the human-score column to numeric is what removes the five
    unusable rows: four are blank and one (Malayalam) holds the string
    ``\`19``, which is not a score.
    """
    full, work = {}, {}
    for lang in LANG_ORDER:
        d = pd.read_excel(path, sheet_name=SHEET_MAP[lang])
        d["H"] = pd.to_numeric(d[COL_HUMAN], errors="coerce")
        full[lang] = d
        work[lang] = d.dropna(subset=["H"]).reset_index(drop=True)
    return full, work


full, work = load_sheets(DATA_XLMR)
print(f"Loaded {sum(len(full[l]) for l in LANG_ORDER):,} rows "
      f"across {len(SHEET_MAP)} sheets")

## Step 2 — The Full and Working Bases

The five dropped rows are itemised rather than summarised, so that the
7,000 → 6,995 step is auditable.

In [ ]:
rows = []
print(f"{'Lang':>5}  {'full':>6}  {'working':>8}  {'dropped':>8}")
print("-" * 33)
for lang in LANG_ORDER:
    n_full, n_work = len(full[lang]), len(work[lang])
    rows.append(dict(lang=lang, full=n_full, working=n_work,
                     dropped=n_full - n_work))
    print(f"{lang:>5}  {n_full:>6}  {n_work:>8}  {n_full - n_work:>8}")

total_full = sum(r["full"] for r in rows)
total_work = sum(r["working"] for r in rows)
print("-" * 33)
print(f"{'TOTAL':>5}  {total_full:>6}  {total_work:>8}  {total_full - total_work:>8}")

# ── Cross-verification against the paper ─────────────────────────────────────
assert total_full == 7000, f"full base = {total_full}, expected 7000"
assert total_work == 6995, f"working base = {total_work}, expected 6995"
print(f"\n\u2713 Full base    = {total_full:,} (paper: 7,000)")
print(f"\u2713 Working base = {total_work:,} (paper: 6,995)")

## Step 3 — Which Rows Were Dropped, and Why

Four rows have a blank human score. One Malayalam row holds the literal string
`` `19 ``, which is a data-entry artefact rather than a score. None of them is
missing a COMET value, which matters for Step 5.

In [ ]:
for lang in LANG_ORDER:
    mask = full[lang]["H"].isna()
    if not mask.any():
        continue
    for idx in full[lang].index[mask]:
        raw = full[lang].loc[idx, COL_HUMAN]
        print(f"  {lang} row {idx:>4}: Human_scores={raw!r:>8}  "
              f"COMET={full[lang].loc[idx, COL_COMET_NAT]:.3f}  (COMET present)")

## Step 4 — The Marathi Severity Base

The severity-inversion analysis in notebook 04 conditions on the primary error
severity, so segments labelled `Default` (no annotated error) fall out.

In [ ]:
n_default = int((full["MAR"][COL_SEVERITY] == "Default").sum())
n_sev = len(full["MAR"]) - n_default
print(f"  MAR: {len(full['MAR'])} total - {n_default} Default = {n_sev} with a severity label")

assert n_sev == 1258, f"MAR severity base = {n_sev}, expected 1258"
print(f"\n\u2713 MAR severity base = {n_sev:,} (paper: 1,258)")

## Step 5 — COMET Completeness

The COMET columns have no missing values in any sheet. This is the reason the
ANOVA in notebook 02 is reported on the full 7,000-segment base even though the
correlation analyses use 6,995: dropping a row for a missing *human* score has
no bearing on a test that only involves COMET.

In [ ]:
print(f"{'Lang':>5}  {'COMET NaN':>10}  {'COMET_rom NaN':>14}  {'H NaN':>6}")
print("-" * 42)
for lang in LANG_ORDER:
    d = full[lang]
    print(f"{lang:>5}  {d[COL_COMET_NAT].isna().sum():>10}  "
          f"{d[COL_COMET_ROM].isna().sum():>14}  {d['H'].isna().sum():>6}")

n_comet_missing = sum(full[l][COL_COMET_NAT].isna().sum() for l in LANG_ORDER)
assert n_comet_missing == 0, "COMET column is not complete"
print("\n\u2713 COMET is complete across all 7,000 segments")

## Step 6 — Save

In [ ]:
out = pd.DataFrame(rows).set_index("lang")
out.loc["TOTAL"] = out.sum()
out["mar_severity_base"] = ""
out.loc["MAR", "mar_severity_base"] = n_sev

path = TABLES_DIR / "sample_sizes.csv"
out.to_csv(path)
print(out.to_string())
print(f"\nSaved \u2192 {path}")

## Step 7 — Output Manifest

In [ ]:
print("=== Notebook 01 — output manifest ===")
for f in sorted(TABLES_DIR.glob("sample_sizes*.csv")):
    print(f"  {f.name}")

## References

**This work.**
Anonymous (2026). *Under review.*

**Information Parity (IP).**
Tsvetkov, A., & Kipnis, A. (2024). Information Parity: Measuring and Predicting the
Multilingual Capabilities of Language Models. *Findings of EMNLP 2024*, pp. 7971–7989.

**Tokenization Parity and tokenizer unfairness.**
Petrov, A., La Malfa, E., Torr, P. H. S., & Bibi, A. (2023). Language Model Tokenizers
Introduce Unfairness Between Languages. *NeurIPS 36*.

**COMET.**
Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural Framework for
MT Evaluation. *EMNLP 2020*, pp. 2685–2702. https://aclanthology.org/2020.emnlp-main.213

**IndicMT Eval dataset.**
Sai B., A., Dixit, T., Nagarajan, V., Kunchukuttan, A., Kumar, P., Khapra, M. M., &
Dabre, R. (2023). IndicMT Eval: A Dataset to Meta-Evaluate Machine Translation Metrics
for Indian Languages. *ACL 2023*, pp. 14210–14228. https://aclanthology.org/2023.acl-long.795

**WMT24 Latin-script controls.**
Kocmi, T., et al. (2024). Findings of the WMT24 General Machine Translation Shared Task.
*Proceedings of WMT 2024*, pp. 1–46. https://aclanthology.org/2024.wmt-1.1